# 06 Labels and cohorts

**What this does.** Keeps every human grade of a target-comp pair, under the rubric it was given with, and serves
them to the other components as named cohorts (`scripts/cohorts.py`). The pair ledger counts every pair a person or
a model judged, with the dates and rubric or prompt versions.

| cohort | who | rubric |
|---|---|---|
| `dev_293` (default) | Barbara | rounds 2-3 old 0/1/2 form |
| `round1_36` | Riya, Barbara, Matthew | round 1 (majority of three) |
| `matthew_v2_262` | Matthew | rubric v2: four yes/no questions, score derived |
| `final_holdout_24` | Barbara | direct-peer Yes/No; **locked** to 10_final_holdout |

A blank grade is never a 0. Barbara's labels are one reviewer's judgment and are pending Michael's review.

**Changelog**
- 2026-11-18: created from `v1/notebooks/golden_set.ipynb`, `golden_set_showcase.ipynb`, Matthew's rubric v2
  (`matt/rubric-yes-no`) and the pair ledger. Added `cohorts.py`. All label tables rebuild exactly.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [2]:
import json, glob, cohorts
pd.Series(cur["labels"])

default_cohort                                 dev_293
cohorts           [dev_293, round1_36, matthew_v2_262]
locked_cohorts                      [final_holdout_24]
dtype: object

## 2. Inputs

In [3]:
L = "outputs-local/v1a/06_labels/"
nb.inputs([L + "barbara_293/sonnet_label.xlsx", L + "barbara_293/review/pairs.csv", L + "round1/reviewer_labels_20261028.json",
           L + "round2/round2_sonnet_sheet.csv", L + "round3/round3_sheet.csv", L + "round3/round3_sheet_v2.csv",
           L + "round3/round3_scores_v2.csv"])

,input,rows,sha256,status
0,outputs-local/v1a/06_labels/barbara_293/sonnet_label.xlsx,NaN,232faa865e0b,matches seed
1,outputs-local/v1a/06_labels/barbara_293/review/pairs.csv,293.0,3ea5583a922b,matches seed
2,outputs-local/v1a/06_labels/round1/reviewer_labels_20261028.json,NaN,ee07b1c14fee,matches seed
3,outputs-local/v1a/06_labels/round2/round2_sonnet_sheet.csv,31.0,6ce91938cbd6,matches seed
4,outputs-local/v1a/06_labels/round3/round3_sheet.csv,262.0,744d67635604,matches seed
5,outputs-local/v1a/06_labels/round3/round3_sheet_v2.csv,262.0,228c924cfe5d,matches seed
6,outputs-local/v1a/06_labels/round3/round3_scores_v2.csv,262.0,65b55cbcf7c8,matches seed


## 3. Results

### 3.1 The cohorts

In [4]:
cohorts.table()

,cohort,who,rubric,pairs,targets,grade 2,grade 1,grade 0,not graded
0,dev_293,Barbara,rounds 2-3 old 0/1/2 form,293.0,45.0,24.0,178.0,91.0,0.0
1,round1_36,"Riya, Barbara, Matthew (majority of 3)","round 1: 0/1/2, blank = 0 with one of five reasons",36.0,2.0,4.0,7.0,21.0,4.0
2,matthew_v2_262,Matthew,"rubric v2: four yes/no questions, score derived",262.0,40.0,60.0,124.0,78.0,0.0
3,final_holdout_24,Barbara,direct-peer Yes/No (Yes = 2; No is not a 0) [locked],NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Every pair judged, by stage (pair ledger)

In [5]:
ledger = Path(sorted(glob.glob(str(paths.out("06_labels") / "pair_ledger" / "2*")))[-1])
long = pd.read_csv(ledger / "judgments.csv", dtype={"target_id": str, "comp_id": str})
j = long[long["judged"]]
print("ledger run:", ledger.name)
pd.DataFrame({stage: {"sources": g["source"].nunique(), "judgments": len(g),
                      "unique pairs": len(g[["target_id", "comp_id"]].drop_duplicates()),
                      "targets": g["target_id"].nunique()} for stage, g in j.groupby("stage")}).T

ledger run: 20261119T014543Z


,sources,judgments,unique pairs,targets
agent_loop,18,1153,365,47
human_review,7,687,353,47
llm_judge,2,2275,2275,201
llm_screen,4,12326,5698,373


### 3.3 Rubric and prompt spans

In [6]:
spans = pd.read_csv(ledger / "spans.csv")
spans[["stage", "rubric", "who", "first", "last", "judgments", "unique_pairs_judged", "targets", "sent_not_judged"]].fillna("")

,stage,rubric,who,first,last,judgments,unique_pairs_judged,targets,sent_not_judged
0,llm_screen,binary screen prompt 2026-10-25a,claude-haiku-5-5,2026-10-27,2026-10-27,4996,4996,373,
1,human_review,"round 1: 0/1/2, blank = 0 with one of five reasons",Barbara;Matthew;Riya,2026-10-28,2026-10-28,108,36,2,
2,llm_judge,"cited grades, 0/1/2 (v0 holdout)",GPT-6.1 Sol,2026-10-28,2026-10-28,1982,1982,156,
3,agent_loop,agent prompt 2026-11-05a,sonnet agent,2026-11-05,2026-11-06,505,341,47,
4,agent_loop,agent prompt 2026-11-05b,sonnet agent,2026-11-05,2026-11-05,146,71,7,
5,llm_screen,binary screen prompt 2026-11-04a,claude-haiku-5-5;claude-sonnet-5-5,2026-11-05,2026-11-19,7330,3148,373,
6,human_review,"rounds 2-3 old form: 0/1/2, six reasons, multi-business rule",Barbara,2026-11-10,2026-11-10,293,293,45,human:round2_sheet:reviewer_a (67);human:round2_sheet:reviewer_b (67);human:round2_she...
7,llm_judge,"source-cited one-shot judge, 0/1/2",GPT-6.1 Sol,2026-11-10,2026-11-10,293,293,45,
8,agent_loop,agent prompt 2026-11-05b + facet scores,sonnet agent,2026-11-12,2026-11-18,502,313,45,
9,human_review,"rubric v2: four yes/no questions, score derived",Matthew,2026-11-13,2026-11-13,262,262,40,


### 3.4 Round 1: three reviewers on the same 36 pairs

In [7]:
r1 = cohorts.load("round1_36")
s = r1[["score_riya", "score_barbara", "score_matthew"]]
print("unanimous:", int((s.nunique(axis=1) == 1).sum()), "of 36; with a majority grade:", int(r1["grade"].notna().sum()))
names = ["riya", "barbara", "matthew"]
pd.DataFrame([{"pair": f"{a} vs {b}", "exact": round((s[f"score_{a}"] == s[f"score_{b}"]).mean(), 3),
               "peer vs non-peer": round(((s[f"score_{a}"] >= 1) == (s[f"score_{b}"] >= 1)).mean(), 3)}
              for i, a in enumerate(names) for b in names[i + 1:]])

unanimous: 11 of 36; with a majority grade: 32


,pair,exact,peer vs non-peer
0,riya vs barbara,0.333,0.611
1,riya vs matthew,0.333,0.500
2,barbara vs matthew,0.833,0.889


### 3.5 Round 3: Barbara (old form) and Matthew (rubric v2) on the same 262 pairs

In [8]:
dev, m2 = cohorts.load("dev_293"), cohorts.load("matthew_v2_262")
both = dev.merge(m2, on=["target_id", "comp_id"], suffixes=("_barbara", "_matthew"))
print(f"{len(both)} shared pairs; exact {(both.grade_barbara == both.grade_matthew).mean():.0%}, "
      f"peer vs non-peer {(both.peer_barbara == both.peer_matthew).mean():.0%}")
display(pd.crosstab(both["grade_barbara"], both["grade_matthew"], rownames=["Barbara"], colnames=["Matthew v2"]))
q = ["same_product_matthew", "same_customers_matthew", "same_revenue_model_matthew", "same_chain_step_matthew"]
(m2[q] == "True").mean().round(2).rename("yes rate (Matthew)").to_frame().T

262 shared pairs; exact 51%, peer vs non-peer 65%


Matthew v2,0.0,1.0,2.0
Barbara,,,
0,36,48,1
1,42,76,38
2,0,0,21


,same_product_matthew,same_customers_matthew,same_revenue_model_matthew,same_chain_step_matthew
yes rate (Matthew),0.24,0.7,0.79,0.77


### 3.6 Barbara's 293 and the agent loop's grade (saved beside her labels)

In [9]:
pd.Series(json.loads((paths.out("06_labels") / "barbara_293" / "review" / "summary.json").read_text()))

cohort              Barbara single reviewer; not consensus gold
pairs                                                       293
targets                                                      45
agent_scored                                                292
exact_agreement                                        0.839041
binary_agreement                                       0.873288
disagreements                                                47
dtype: object

## 4. Checks

In [10]:
import contextlib, io, tempfile
import review_labels, round3_sheet, reviewer_gold_eval, rubric_v2
def same(a, b, **kw):
    x, y = pd.read_csv(a, dtype=str, **kw).fillna(""), pd.read_csv(b, dtype=str, **kw).fillna("")
    return x.equals(y)
R = paths.out("06_labels")
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    with contextlib.redirect_stdout(io.StringIO()):
        review_labels.main(["--out", str(tmp / "review")])
        round3_sheet.main(["--candidates", str(paths.path(cur["retrieval"]["candidates"])), "--out", str(tmp / "r3")])
        reviewer_gold_eval.run(reviewer_gold_eval.LABELS, reviewer_gold_eval.POOL, tmp / "rge")
        rubric_v2.main(["scores", str(R / "round3" / "round3_sheet_v2.csv"), str(tmp / "v2.csv")])
    checks.add("Barbara's 293 table rebuilds from her workbook",
               all(same(tmp / "review" / f, R / "barbara_293" / "review" / f) for f in ["pairs.csv", "confusion.csv", "disagreements.csv"]))
    checks.add("round-3 sheet and pairs rebuild (seed 2026)", same(tmp / "r3" / "round3_sheet.csv", R / "round3" / "round3_sheet.csv")
               and same(tmp / "r3" / "round3_pairs.csv", R / "round3" / "round3_pairs.csv"))
    checks.add("Matthew's rubric-v2 scores re-derive from his answers",
               same(tmp / "v2.csv", R / "round3" / "round3_scores_v2.csv", encoding="utf-8-sig"))
    v1_rge = sorted((paths.V1_OUTPUTS / "golden_set" / "reviewer_evaluation").glob("*.csv"))
    checks.add("round-1 reviewer evaluation reproduces v1", len(v1_rge) == 9 and all(same(tmp / "rge" / f.name, f) for f in v1_rge),
               f"{len(v1_rge)} tables")
c = cohorts.table(include_locked=True)
checks.add("cohort sizes (293 / 36 / 262 / 24)", c["pairs"].tolist() == [293, 36, 262, 24])
try:
    cohorts.load("final_holdout_24"); locked = False
except cohorts.LockedCohort:
    locked = True
checks.add("final_holdout_24 refuses to load outside 10", locked)
old = pd.read_csv(R / "pair_ledger" / "20261118T205008Z" / "sources.csv")
new = pd.read_csv(ledger / "sources.csv")
m = old.merge(new, on="source", suffixes=("", "_new"))
cols = ["pairs", "judged", "grade_2", "grade_1", "grade_0", "sha256"]
checks.add("ledger: the 33 v1 sources unchanged since 11-18", len(m) == 33 and all((m[k] == m[k + "_new"]).all() for k in cols),
           f"{len(new)} sources now")
checks.show()

7/7 checks pass


,check,result,detail
0,Barbara's 293 table rebuilds from her workbook,pass,
1,round-3 sheet and pairs rebuild (seed 2026),pass,
2,Matthew's rubric-v2 scores re-derive from his answers,pass,
3,round-1 reviewer evaluation reproduces v1,pass,9 tables
4,cohort sizes (293 / 36 / 262 / 24),pass,
5,final_holdout_24 refuses to load outside 10,pass,
6,ledger: the 33 v1 sources unchanged since 11-18,pass,34 sources now


## 5. History (v1)

- **Round 1 (10-27/28):** Michael's sheet, then Riya, Barbara and Matthew on 36 PCTY/WEN pairs; 11 unanimous.
- **Round 2 (11-04):** a 67-pair sheet for CSL, AFG, AZZ, BBCP, BUKS + 2 WEN, three neutral reviewer slots, the six
  reasons and the multi-business rule (`GOLDEN_SET_ROUND2.md`). Never filled; superseded by the Sonnet-list rounds.
- **Rounds 2-3 on the Sonnet lists (11-05/06):** 31 + 262 pairs; Barbara graded all 293 (committed 11-10).
- **Rubric v2 (11-13, Matthew and Riya):** four yes/no questions, score derived (`RUBRIC_V2.md`); Matthew answered the
  262 round-3 pairs. Question 2 (same customers) decides most disagreements with Barbara's old form.
- **Barbara's direct peers (11-17):** 90 pairs she picked; the 24 with comps in the universe are the locked final
  holdout; the 66 with comps outside the universe are left out of v1a.
- `integration_sources.py` and Barbara's published-list scripts stay in v1 (they read files outside the repo); their
  outputs are seeded here as frozen sources (`barbara_293/sonnet_label.xlsx`, `barbara_published/`).